# 15 — Post-hoc reviewer analyses

**Status: exploratory/post-hoc.** The primary TEST result and SRNet secondary TEST
result were already observed before this notebook was designed. Nothing in this
notebook may be used to change the frozen allocator, alpha, detector or primary claim.

It adds two reviewer-facing diagnostics at the frozen primary payload:

\[
R=0.009\ \text{net bpp}.
\]

1. sensitivity over \(\alpha\in\{0,0.25,0.5,0.75,1\}\);
2. a same-codec Sobel-gradient complexity baseline.


In [ ]:
from pathlib import Path
import json, joblib, os, time, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdhlab.io import read_gray
from rdhlab.pipeline import run_frozen_image_precomputed
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.transfer import paired_method_bootstrap
from rdhlab.detector_repair import load_enhanced_cnn, score_enhanced_residual_cnn
from rdhlab.freeze_protocol import sha256_file
from rdhlab.posthoc_revision_v15 import alpha_order, gradient_complexity_order, stable_id_hash

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=max(5000,int(config['statistics'].get('bootstrap_resamples',5000)))

manifest=pd.read_csv(config['dataset']['prepared_manifest'])
test=manifest[manifest.split=='test'].reset_index(drop=True)
test['source_id']=test.source_id.astype(str)
id_to_path=dict(zip(test.source_id,test.path))

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
assert np.isclose(float(allocator['alpha']),0.25)
primary_bpp=float(allocator['teacher_payload_bpp'])
assert np.isclose(primary_bpp,0.009)

out06=Path('/workspace/results/frozen_test_final')
protocol06=json.loads((out06/'test_protocol.json').read_text())
common=pd.read_csv(out06/'common_feasible_ids.csv')
common['source_id']=common.source_id.astype(str)
primary_ids=common[np.isclose(common.target_net_bpp.astype(float),primary_bpp)].source_id.tolist()
assert len(primary_ids)==1968

per06=pd.read_csv(out06/'per_image.csv')
per06['source_id']=per06.source_id.astype(str)
context_dir=out06/'contexts'/protocol06['context_tag']

out07=Path('/workspace/results/final_steganalysis')
cover_scores=pd.read_csv(out07/'cover_scores.csv')
cover_scores['source_id']=cover_scores.source_id.astype(str)
cover_map=dict(zip(cover_scores.source_id,cover_scores.cover_score.astype(float)))
existing=pd.read_csv(out07/'enhanced_cnn_test_scores.csv')
existing['source_id']=existing.source_id.astype(str)

cnn_path=Path('/workspace/results/models/enhanced_residual_cnn_05e.pt')
cnn,device=load_enhanced_cnn(cnn_path)

OUT=Path('/workspace/results/posthoc_revision_v15')
CK=OUT/'checkpoints'
OUT.mkdir(parents=True,exist_ok=True); CK.mkdir(parents=True,exist_ok=True)

protocol={
  'analysis_status':'POST_HOC_EXPLORATORY_REVISION_V15',
  'created_after_primary_test_result':True,
  'created_after_srnet_secondary_test_result':True,
  'primary_payload_bpp':primary_bpp,
  'alpha_grid':[0.0,0.25,0.5,0.75,1.0],
  'complexity_baseline':'descending block mean Sobel magnitude',
  'primary_source_ids_sha256':stable_id_hash(primary_ids),
  'frozen_alpha':0.25,
  'frozen_detector_sha256':sha256_file(cnn_path),
  'frozen_allocator_sha256':sha256_file(allocator_path),
  'no_retuning_permitted':True,
  'interpretation':'exploratory sensitivity and controlled baseline only'
}
(OUT/'posthoc_revision_v15_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(json.dumps(protocol,indent=2))
print('CNN device:',device)


In [ ]:
def load_ctx(sid):
    ctx=joblib.load(context_dir/f'{sid}.joblib')
    if ctx.get('context_tag')!=protocol06['context_tag']:
        raise RuntimeError(f'stale context {sid}')
    return ctx['block_rows'],ctx['plans']

def atomic_csv(df,path):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False); os.replace(tmp,path)

def _checkpoint_prefix(path):
    path=Path(path)
    if not path.exists(): return pd.DataFrame()
    df=pd.read_csv(path); df['source_id']=df.source_id.astype(str)
    if len(df)>len(primary_ids) or df.source_id.tolist()!=primary_ids[:len(df)]:
        raise RuntimeError(f'invalid resume prefix: {path}')
    return df

def score_generated(name,order_builder,batch_size=16):
    ck=CK/f'{name}.csv'
    df=_checkpoint_prefix(ck)
    start=len(df)
    rows=df.to_dict('records') if start else []
    t0=time.perf_counter(); imgs=[]; metas=[]

    def flush():
        nonlocal imgs,metas,rows
        if not imgs: return
        ss=score_enhanced_residual_cnn(cnn,imgs,device=device,batch_size=batch_size)
        for m,s in zip(metas,ss):
            cs=float(cover_map[m['source_id']])
            rows.append({**m,'cover_score':cs,'stego_score':float(s),'score_delta':float(s-cs)})
        imgs=[]; metas=[]
        atomic_csv(pd.DataFrame(rows),ck)

    print(name,'resume',start,'/',len(primary_ids))
    for pos in range(start,len(primary_ids)):
        sid=primary_ids[pos]
        x=read_gray(id_to_path[sid]); br,plans=load_ctx(sid)
        order=order_builder(x,br)
        rr=run_frozen_image_precomputed(
            x,sid,primary_bpp,name,{name:order},br,bs,seed,False,None,plans=plans
        )
        if not rr['feasible']:
            flush()
            rows.append({'source_id':sid,'strategy':name,'feasible':False})
            atomic_csv(pd.DataFrame(rows),ck)
        else:
            if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
                raise RuntimeError(f'reversibility failed {sid} {name}')
            if int(rr['net_payload_bits'])!=int(rr['target_net_bits']):
                raise RuntimeError(f'net payload mismatch {sid} {name}')
            imgs.append(rr['stego'])
            metas.append({
              'source_id':sid,'strategy':name,'feasible':True,
              'target_net_bpp':primary_bpp,'actual_net_bpp':rr['actual_net_bpp'],
              'net_payload_bits':rr['net_payload_bits'],'target_net_bits':rr['target_net_bits'],
              'psnr':rr['psnr'],'ssim':rr['ssim'],
              'sideinfo_bits':rr['sideinfo_bits'],'gross_payload_bits':rr['gross_payload_bits'],
              'used_blocks':rr['used_blocks'],'changed_pixels':rr['changed_pixels']
            })
            if len(imgs)>=batch_size: flush()
        if (pos+1)%250==0 or pos+1==len(primary_ids):
            flush(); print(name,pos+1,'/',len(primary_ids),'elapsed',f'{(time.perf_counter()-t0)/60:.1f} min',flush=True)
    flush()
    out=pd.DataFrame(rows)
    if len(out)!=len(primary_ids) or out.source_id.astype(str).tolist()!=primary_ids:
        raise RuntimeError(f'{name}: output alignment failure')
    return out

def existing_strategy(strategy):
    z=existing[
        (existing.strategy.astype(str)==strategy) &
        np.isclose(existing.target_net_bpp.astype(float),primary_bpp)
    ].copy()
    z=z.set_index('source_id').reindex(primary_ids).reset_index()
    if z.stego_score.isna().any(): raise RuntimeError(f'missing frozen scores for {strategy}')
    q=per06[
        (per06.strategy.astype(str)==strategy) &
        np.isclose(per06.target_net_bpp.astype(float),primary_bpp)
    ].set_index('source_id').reindex(primary_ids)
    z['strategy']=strategy; z['feasible']=True
    for col in ['actual_net_bpp','net_payload_bits','target_net_bits','psnr','ssim','sideinfo_bits','gross_payload_bits','used_blocks','changed_pixels']:
        z[col]=q[col].to_numpy()
    return z

alpha_frames={
  0.0:existing_strategy('detectability'),
  0.25:existing_strategy('joint'),
  1.0:existing_strategy('predictability'),
}
for a in [0.5,0.75]:
    name=f'alpha_{str(a).replace(".","p")}'
    alpha_frames[a]=score_generated(name,lambda x,br,a=a: alpha_order(br,a))

complexity=score_generated('gradient_complexity',lambda x,br: gradient_complexity_order(x,bs)[0])
print('Generated new post-hoc sensitivity and complexity cases.')


In [ ]:
def summarize_method(df,label):
    g=df[df.get('feasible',False).astype(bool)].copy() if 'feasible' in df.columns else df.copy()
    g=g[g.stego_score.notna()].copy()
    c=g.cover_score.to_numpy(float); s=g.stego_score.to_numpy(float)
    y=np.tile([0,1],len(g)); sc=np.column_stack([c,s]).reshape(-1)
    m=detector_metrics(y,sc,fixed_fpr)
    ci=paired_detector_bootstrap(c,s,fixed_fpr,n_resamples=n_boot,confidence=confidence,seed=seed+15000+len(label))
    return {
      'method':label,'n':len(g),'feasible_fraction':len(g)/len(primary_ids),
      'actual_net_bpp_mean':float(g.actual_net_bpp.mean()),
      'psnr_mean':float(g.psnr.mean()),'ssim_mean':float(g.ssim.mean()),
      'sideinfo_fraction_gross_mean':float((g.sideinfo_bits/g.gross_payload_bits).mean()),
      'used_blocks_mean':float(g.used_blocks.mean()),
      'auc':float(m['auc']),'auc_ci_low':float(ci['auc_low']),'auc_ci_high':float(ci['auc_high']),
      'tpr_at_5pct_fpr':float(m['tpr_at_fpr']),'tpr_ci_low':float(ci['tpr_low']),'tpr_ci_high':float(ci['tpr_high']),
      'score_delta_mean':float((s-c).mean())
    }

alpha_rows=[]
for a in [0.0,0.25,0.5,0.75,1.0]:
    r=summarize_method(alpha_frames[a],f'alpha={a:g}'); r['alpha']=a; alpha_rows.append(r)
alpha_summary=pd.DataFrame(alpha_rows).sort_values('alpha')
alpha_summary.to_csv(OUT/'alpha_sensitivity_primary_cnn.csv',index=False)
display(alpha_summary)

pred=alpha_frames[1.0].set_index('source_id')
pairs=[]
for a in [0.0,0.25,0.5,0.75]:
    cur=alpha_frames[a].set_index('source_id')
    ids=[sid for sid in primary_ids if sid in cur.index and sid in pred.index and bool(cur.loc[sid,'feasible']) and bool(pred.loc[sid,'feasible'])]
    c=np.asarray([cover_map[sid] for sid in ids],float)
    aa=cur.loc[ids].stego_score.to_numpy(float); bb=pred.loc[ids].stego_score.to_numpy(float)
    q=paired_method_bootstrap(c,aa,bb,fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,seed=seed+15100+int(a*100))
    q.update({'alpha':a,'reference_alpha':1.0,'pairs':len(ids),'analysis_status':'POST_HOC_EXPLORATORY'})
    pairs.append(q)
pair_df=pd.DataFrame(pairs)
pair_df.to_csv(OUT/'alpha_sensitivity_pairwise_vs_predictability.csv',index=False)
display(pair_df)

comp_summary=pd.DataFrame([
    summarize_method(alpha_frames[0.25],'joint alpha=0.25'),
    summarize_method(complexity,'gradient complexity'),
    summarize_method(alpha_frames[1.0],'predictability alpha=1'),
])
comp_summary.to_csv(OUT/'complexity_baseline_primary_cnn.csv',index=False)
display(comp_summary)

cx=complexity.set_index('source_id'); j=alpha_frames[0.25].set_index('source_id')
ids=[sid for sid in primary_ids if sid in cx.index and bool(cx.loc[sid,'feasible']) and bool(j.loc[sid,'feasible'])]
c=np.asarray([cover_map[sid] for sid in ids],float)
jscore=j.loc[ids].stego_score.to_numpy(float); cxscore=cx.loc[ids].stego_score.to_numpy(float)
pcomp=paired_method_bootstrap(c,jscore,cxscore,fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,seed=seed+15200)
pcomp.update({
  'method':'joint alpha=0.25','reference':'gradient complexity','pairs':len(ids),
  'analysis_status':'POST_HOC_EXPLORATORY_CONTROLLED_COMPLEXITY_BASELINE',
  'baseline_definition':'descending mean Sobel gradient magnitude per 64x64 block',
  'not_a_reimplementation_of_Hong_or_Chen':True
})
(OUT/'complexity_baseline_paired.json').write_text(json.dumps(pcomp,indent=2),encoding='utf-8')
print(json.dumps(pcomp,indent=2))


In [ ]:
fig,ax=plt.subplots(figsize=(6.4,4.5))
ax.plot(alpha_summary.alpha,alpha_summary.auc,marker='o')
ax.set_xlabel(r'$\\alpha$'); ax.set_ylabel('Frozen enhanced-CNN ROC-AUC')
ax.set_title('Post-hoc TEST sensitivity at 0.009 net bpp'); ax.grid(True,alpha=.2)
fig.tight_layout(); fig.savefig(OUT/'alpha_sensitivity_auc.png',dpi=300); plt.show()

fig,ax=plt.subplots(figsize=(6.4,4.5))
ax.plot(alpha_summary.alpha,alpha_summary.psnr_mean,marker='o')
ax.set_xlabel(r'$\\alpha$'); ax.set_ylabel('Mean PSNR [dB]')
ax.set_title('Post-hoc distortion sensitivity at 0.009 net bpp'); ax.grid(True,alpha=.2)
fig.tight_layout(); fig.savefig(OUT/'alpha_sensitivity_psnr.png',dpi=300); plt.show()

complete={
  'status':'COMPLETE','analysis_status':'POST_HOC_EXPLORATORY_REVISION_V15',
  'primary_results_unchanged':True,'allocator_retuned':False,
  'alpha_grid':[0.0,0.25,0.5,0.75,1.0],
  'complexity_baseline':'gradient complexity','primary_payload_bpp':primary_bpp,
  'primary_source_ids':len(primary_ids),'no_retuning_permitted':True
}
(OUT/'posthoc_revision_v15_complete.json').write_text(json.dumps(complete,indent=2),encoding='utf-8')
print(json.dumps(complete,indent=2)); print('PATCH 15 COMPLETE')
